# Global Gun Violence & Civilian Firearms (2000–2024)
## Econometric Panel Identification, Geospatial Disparities & Country Archetype Segmentation

> **ElmasCore Analytical Monograph** • Grounded in UNODC, Small Arms Survey (2018), and World Bank WDI.

---

### Research Agenda
1. **The Empirical Paradox**: What is the true bivariate and multivariate relationship between civilian firearm penetration (`firearms_per_100_people`) and firearm homicide rates across 95 nations?
2. **Panel Identification**: Once unobserved country heterogeneity and global macroeconomic time shocks are absorbed via **Two-Way Fixed Effects PanelOLS**, what are the macro determinants of firearm homicides?
3. **Global Archetypes**: How do countries cluster into structural regimes across gun ownership, homicide severity, and socio-economic inequality?

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.core.config import AnalysisConfig
from src.data_pipeline import load_raw_datasets, build_panel_dataset
from src.econometrics import estimate_panel_ols, compute_spearman_rank_correlation
from src.segmentation import segment_country_archetypes
from src.modeling import analyze_risk_factors
from src.visualization import render_world_choropleth, render_quadrant_scatter

print('Environment and GGVCF analytical modules loaded successfully!')

## 1. Data Ingestion & The Empirical Paradox

We load the three foundational tables: `civilian_firearms_2017.csv`, `country_gun_profile.csv`, and `firearm_homicides_country_year.csv`.

In [2]:
cfg = AnalysisConfig()
civilian_df, profile_df, panel_df = load_raw_datasets(cfg.paths)
print(f'Country profiles: {len(profile_df)} countries')
print(f'Panel observations: {len(panel_df)} country-years')

# Compute Spearman rank correlation across 95 countries
rho, pval = compute_spearman_rank_correlation(
    profile_df,
    'firearms_per_100_people',
    'avg_2013_2022_firearm_homicide_rate'
)
print(f'Spearman rank correlation: rho = {rho:.4f} (p-value = {pval:.4f})')

## 2. Two-Way Fixed Effects Econometric Estimation

We estimate:
$$\ln(\text{Firearm Homicides}_{it}) = \beta_1 \ln(\text{GDP pc}_{it}) + \beta_2 \text{Gini}_{it} + \beta_3 \text{Urban}_{it} + \alpha_i + \gamma_t + \varepsilon_{it}$$
where $\alpha_i$ absorbs all time-invariant country-specific institutions and $\gamma_t$ absorbs global macro shocks.

In [3]:
bundle = build_panel_dataset(panel_df, cfg.econometrics)
reg_report = estimate_panel_ols(bundle, cfg.econometrics)
print(reg_report.summary_text)

## 3. Global Country Archetypes (PCA + K-Means)

Unsupervised segmentation into 4 global structural regimes.

In [4]:
cluster_report = segment_country_archetypes(profile_df, cfg.clustering)
print(f'Silhouette Score: {cluster_report.silhouette_score:.3f}')
print('Cluster Means:')
cluster_report.cluster_means

## 4. Interactive Visualizations

Displaying the interactive quadrant plot and global choropleth.

In [5]:
fig_quadrant = render_quadrant_scatter(cluster_report.profile_df)
fig_quadrant.show()